# 🖊️ Conditional VAE — MNIST Digit Generator
**Train from scratch → Deploy as public Gradio app**

Steps:
1. Install deps & train CVAE model (~25 min on T4)
2. Launch Gradio app with `share=True` for public URL
3. (Optional) Push to HuggingFace Spaces for permanent hosting

> Make sure Runtime → Change runtime type → **T4 GPU** is selected!

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Write training script to disk ────────────
train_script = '''
# [PASTE FULL CONTENT OF train_cvae_mnist.py HERE]
# OR use: !wget https://raw.githubusercontent.com/YOUR_REPO/train_cvae_mnist.py
'''
# If using the .py file directly:
# !wget -q https://your-gist-url/train_cvae_mnist.py
print('Script ready.')

In [ ]:
# ── Cell 3: Train the model ──────────────────────────
# This cell assumes train_cvae_mnist.py is in the current directory
%run train_cvae_mnist.py
# Training takes ~20-25 min on T4 GPU for 60 epochs
# Watch for loss decreasing: target recon_loss < 80, kl_loss < 15

In [ ]:
# ── Cell 4: Preview generated samples ───────────────
from IPython.display import Image as IPImage, display
import os

# Show the final epoch grid
sample_files = sorted([f for f in os.listdir('samples') if f.startswith('epoch')])
if sample_files:
    display(IPImage(f'samples/{sample_files[-1]}'))

# Show per-digit test images
for d in range(10):
    path = f'samples/test_digit_{d}.png'
    if os.path.exists(path):
        print(f'Digit {d}:')
        display(IPImage(path))

In [ ]:
# ── Cell 5: Launch Gradio app (public link) ──────────
!pip install gradio -q

# Assumes app.py is in current dir (or paste contents inline)
# %run app.py  ← doesn't work in notebooks for long-running servers

# Instead, launch inline:
import subprocess, threading, time

# Quick inline demo without running full app.py:
import gradio as gr
import torch
from torchvision.utils import make_grid
from PIL import Image
import numpy as np

# (model must already be loaded from training above)
# If not, reload:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Re-import model classes (they should be in scope from %run)
model_demo = CVAE().to(DEVICE)
model_demo.load_state_dict(torch.load('cvae_mnist.pth', map_location=DEVICE))
model_demo.eval()
print('Model loaded for demo.')

def generate_demo(digit_str, temperature):
    digit = int(digit_str)
    labels = torch.full((5,), digit, dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        z = torch.randn(5, 128, device=DEVICE) * temperature
        imgs = model_demo.decoder(z, labels).cpu()
    
    # Build grid
    grid = make_grid(imgs, nrow=5, padding=4, normalize=False)
    arr = (grid.permute(1,2,0).numpy() * 255).astype(np.uint8)
    if arr.shape[2] == 1:
        arr = arr[:,:,0]
    # Upscale
    pil = Image.fromarray(arr).resize(
        (arr.shape[1]*6, arr.shape[0]*6), Image.NEAREST
    )
    return pil

iface = gr.Interface(
    fn=generate_demo,
    inputs=[
        gr.Dropdown([str(i) for i in range(10)], value='7', label='Digit (0-9)'),
        gr.Slider(0.3, 1.5, value=0.85, step=0.05, label='Diversity')
    ],
    outputs=gr.Image(type='pil', label='5 Generated Samples'),
    title='✍ Handwritten Digit Generator',
    description='Conditional VAE trained on MNIST from scratch. Each click = new unique samples!',
    examples=[['0', 0.8], ['3', 0.9], ['7', 0.85], ['9', 1.1]],
)

iface.launch(share=True)  # ← Gives you a public https://xxxxx.gradio.live link!

In [ ]:
# ── Cell 6 (Optional): Deploy to HuggingFace Spaces ──
# Prerequisites:
#   1. Create account at huggingface.co
#   2. Create new Space: https://huggingface.co/new-space
#      SDK: Gradio, Hardware: CPU Basic (free)
#   3. Get your HF token from: https://huggingface.co/settings/tokens

HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # replace with your token
HF_REPO  = 'YOUR_USERNAME/digit-generator'  # replace with your repo

!pip install huggingface_hub -q

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

# Upload files
for fname in ['app.py', 'cvae_mnist.pth', 'requirements.txt']:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=HF_REPO,
        repo_type='space',
        token=HF_TOKEN,
    )
    print(f'Uploaded: {fname}')

print(f'\n🚀 App live at: https://huggingface.co/spaces/{HF_REPO}')